# Stage 5: XGBoost, the model that learns from many clues

**The idea.** GARCH uses one formula. XGBoost is a panel of hundreds of **simple advisors**. Each advisor asks a few yes/no questions about the clues ("Was last month's wiggle above 1.5%? Is volume high today?") and gives the forecast a small nudge. Each new advisor concentrates on what the earlier ones got wrong. Together they form a strong forecaster.

**The big difference from GARCH:** one model learns from **all 1,586 stocks at once**, so it has seen far more storms and calms than any single stock could show it.

**What it predicts:** the *logarithm* of next week's wiggle, so errors count as percentages and a calm stock matters as much as a jumpy one.

Run the cells top to bottom. Kernel: **RiskWise (.venv)**.

In [ ]:
# Cell 1: tools, and the clues table we built (pipeline/features.py)
import sys
sys.path.append("../pipeline")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from features import FEATURE_COLS, make_features
from common import load_prices

tab = pd.read_parquet("../data/features.parquet")
tab["date"] = pd.to_datetime(tab["date"])
print(f"{len(tab):,} rows, {tab['sym'].nunique()} stocks, {tab['date'].min().date()} to {tab['date'].max().date()}")
print("clues:", FEATURE_COLS)

## The clues

Each row is one stock on one day. All clues use only that day and earlier days:
- `yz5, yz10, yz20, yz60, yz120`: the wiggle over the last 5, 10, 20, 60, 120 days (Yang-Zhang, as in Stage 3)
- `sw5`: the simple wiggle of the last 5 days
- `absret1`: how big yesterday's move was
- `range5`: the average high-to-low range of the last 5 days
- `vol_ratio`: is trading volume above or below its normal level?
- `gap_5_60`: is last week wigglier than the stock's usual? (the "waves dying down" idea, which XGBoost has to learn for itself)

They are all in **log** form, so a clue of 0.3 means about 1.35% a day (because exp(0.3) is about 1.35).

In [ ]:
# Cell 2: a few rows for Reliance, with the answer next to the clues
rel = tab[tab["sym"] == "RELIANCE"].set_index("date")
rel[FEATURE_COLS + ["target"]].dropna().tail(4).round(2)

`target` is the log of the **real** wiggle over the next 5 days. It is the answer, not a clue. The model's job: look at the clues, guess the target.

## Step 1: train one model, and look inside

We train on everything up to a cut-off date, keeping the **last year of the training data as a "practice exam"** (a validation slice). XGBoost adds advisors one at a time and we **stop adding when the practice-exam score stops improving**. This is called *early stopping*, and it protects against overfitting: memorising the old data instead of learning the pattern.

We also skip 2 out of 3 days when training, because neighbouring days overlap heavily (their 5-day windows share 4 days), so they add little new information.

In [ ]:
# Cell 3: train one model, with early stopping on a practice-exam slice
CUTOFF = pd.Timestamp("2023-09-26")        # training answers must finish before the real exam starts (2023-10-06)
tab["stride"] = tab.groupby("sym").cumcount() % 3 == 0          # every third day

PARAMS = dict(objective="reg:squarederror", tree_method="hist", max_depth=4, learning_rate=0.05,
              subsample=0.8, colsample_bytree=0.8, min_child_weight=50, n_jobs=-1)

train = tab[(tab["date"] <= CUTOFF) & tab["target"].notna() & tab["stride"]]
practice_start = CUTOFF - pd.DateOffset(years=1)
learn_part = train[train["date"] <= practice_start]
practice_part = train[train["date"] > practice_start]

model = xgb.XGBRegressor(n_estimators=800, early_stopping_rounds=30, **PARAMS)
model.fit(learn_part[FEATURE_COLS], learn_part["target"],
          eval_set=[(practice_part[FEATURE_COLS], practice_part["target"])], verbose=False)
print("rows used to learn:", f"{len(learn_part):,}", "| practice-exam rows:", f"{len(practice_part):,}")
print("advisors (trees) kept:", model.best_iteration + 1)

In [ ]:
# Cell 4: which clues did the advisors rely on most?
importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values()
importance.plot.barh(figsize=(7, 3.5), title="How much each clue mattered")
plt.show()

In [ ]:
# Cell 5: peek at the very first advisor: a few yes/no questions (a tiny flowchart)
print(model.get_booster().get_dump()[0][:900])

Read the dump above: each line is a question such as `[yz20<0.31]` ("is the 20-day log-wiggle below 0.31?") with a **yes** branch and a **no** branch, ending in small numbers called leaves. That number is the advisor's nudge. Hundreds of these, added together, make the forecast.

## Step 2: do it honestly, 6 months at a time (walk forward)

The real exam is the most recent 3 years. We cut it into six 6-month blocks. For each block we re-train **using only data before that block** (with 10 days thrown away at the boundary so no training answer reaches into the block), pick the number of trees on the practice slice, and forecast the block. Then we move to the next block and repeat, so the model keeps learning from the past but never sees the future.

In [ ]:
# Cell 6: the walk-forward loop (about a minute)
EXAM_START = tab["date"].max() - pd.DateOffset(years=3)
edges = [EXAM_START + pd.DateOffset(months=6 * k) for k in range(7)]
edges[-1] = tab["date"].max() + pd.Timedelta(days=1)

blocks = []
for k in range(6):
    start, end = edges[k], edges[k + 1]
    cutoff = start - pd.Timedelta(days=10)                         # boundary gap (purge)
    train = tab[(tab["date"] <= cutoff) & tab["target"].notna() & tab["stride"]]
    practice_start = cutoff - pd.DateOffset(years=1)
    learn_part, practice_part = train[train["date"] <= practice_start], train[train["date"] > practice_start]
    m = xgb.XGBRegressor(n_estimators=800, early_stopping_rounds=30, **PARAMS)
    m.fit(learn_part[FEATURE_COLS], learn_part["target"],
          eval_set=[(practice_part[FEATURE_COLS], practice_part["target"])], verbose=False)
    final = xgb.XGBRegressor(n_estimators=m.best_iteration + 1, **PARAMS).fit(train[FEATURE_COLS], train["target"])
    test = tab[(tab["date"] >= start) & (tab["date"] < end) & tab["actual_next5"].notna()]
    out = test[["sym", "date", "actual_next5", "naive5", "naive20"]].copy()
    out["xgb"] = np.exp(final.predict(test[FEATURE_COLS]))         # back from log to % a day
    out["block"] = k + 1
    blocks.append(out)
    print(f"block {k + 1}: {start.date()} to {end.date()}  trees {m.best_iteration + 1}  exam rows {len(test):,}")
res = pd.concat(blocks)

## Proof that the clues contain no future information

Two checks. First: the answer columns must not be among the clues. Second: chop a stock's history after one date, rebuild its clues, and check the clues for that date **did not change**. If they did, a clue would be peeking.

In [ ]:
# Cell 7: no-peeking checks
assert not any(c in FEATURE_COLS for c in ["target", "actual_next5", "naive5", "naive20"]), "an answer column is among the clues!"

df = load_prices("RELIANCE")
full = make_features(df).dropna(subset=FEATURE_COLS)
d = full.index[-300]
chopped = make_features(df.loc[:d])                      # history ends at d: no future at all
a, b = full.loc[d, FEATURE_COLS].astype(float), chopped.loc[d, FEATURE_COLS].astype(float)
print(pd.DataFrame({"full history": a, "chopped at d": b}).round(4).T.to_string())
assert np.allclose(a.values, b.values), "the clues changed, so they used the future!"
print("\nIdentical. The clues use only the past.")

## The scoreboard

Score = average miss divided by the stock's typical real wiggle, so calm and jumpy stocks count equally. Lower is better. We average over all stocks.

In [ ]:
# Cell 8: the main scoreboard, all 1,586 stocks
def miss_share(col):
    g = res.assign(miss=(res["actual_next5"] - res[col]).abs()).groupby("sym")
    return g["miss"].mean() / g["actual_next5"].median()

share = pd.DataFrame({"Naive": miss_share("naive5"), "Naive-20 (last month)": miss_share("naive20"), "XGBoost": miss_share("xgb")})
print("Average miss as share of the typical wiggle (mean over", len(share), "stocks, lower is better):")
print(share.mean().round(3).to_string())
print("\nXGBoost beats Naive on", round((share["XGBoost"] < share["Naive"]).mean() * 100, 1), "% of stocks")
print("XGBoost beats Naive-20 on", round((share["XGBoost"] < share["Naive-20 (last month)"]).mean() * 100, 1), "% of stocks")
print("XGBoost's gain over Naive:", round((1 - share["XGBoost"].mean() / share["Naive"].mean()) * 100, 1), "%")
print("XGBoost's gain over Naive-20:", round((1 - share["XGBoost"].mean() / share["Naive-20 (last month)"].mean()) * 100, 1), "%")

## Is the gain stable, or a lucky stretch?

If XGBoost only won in one 6-month block, we should not trust it. This table shows each block separately.

In [ ]:
# Cell 9: the same score in each 6-month block
rows = []
for k, part in res.groupby("block"):
    def block_share(col):
        g = part.assign(miss=(part["actual_next5"] - part[col]).abs()).groupby("sym")
        return (g["miss"].mean() / g["actual_next5"].median()).mean()
    rows.append({"block": k, "from": part["date"].min().date(), "Naive": block_share("naive5"),
                 "Naive-20": block_share("naive20"), "XGBoost": block_share("xgb")})
pd.DataFrame(rows).set_index("block").round(3)

## Against GARCH, on the five big stocks

GARCH is fitted one stock at a time, so we compare on the same 5 stocks as Stage 4, with the same scale correction (measured on the study period only) and the same exam dates.

In [ ]:
# Cell 10: GARCH vs XGBoost on the five big stocks
from garch_model import garch_guesses
BIG = ["RELIANCE", "TCS", "INFY", "HDFCBANK", "SBIN"]
rows = []
for s in BIG:
    g = garch_guesses(load_prices(s)).rename("garch_raw")
    t = tab[tab["sym"] == s].set_index("date")[["actual_next5"]].join(g, how="inner").dropna()
    study = t[t.index <= EXAM_START - pd.Timedelta(days=10)]
    scale = (study["actual_next5"] / study["garch_raw"]).median()        # from the study period only
    x = res[res["sym"] == s].set_index("date").join((g * scale).rename("garch_scaled"), how="inner").dropna()
    med = x["actual_next5"].median()
    rows.append({"stock": s, "Naive": (x["actual_next5"] - x["naive5"]).abs().mean() / med,
                 "Naive-20": (x["actual_next5"] - x["naive20"]).abs().mean() / med,
                 "GARCH scaled": (x["actual_next5"] - x["garch_scaled"]).abs().mean() / med,
                 "XGBoost": (x["actual_next5"] - x["xgb"]).abs().mean() / med})
five = pd.DataFrame(rows).set_index("stock")
display(five.round(3))
print("average:", five.mean().round(3).to_dict())

In [ ]:
# Cell 11: SBIN over the last year of the exam: real wiggle vs Naive vs XGBoost
x = res[res["sym"] == "SBIN"].set_index("date").iloc[-250:]
x[["actual_next5", "naive5", "xgb"]].plot(figsize=(10, 4), title="SBIN: real next-5-day wiggle vs two guesses (% per day)")
plt.ylabel("% per day")
plt.show()

In [ ]:
# Cell 12: save XGBoost's exam guesses, so the final scoreboard (next notebooks) can use them
res[["sym", "date", "xgb", "block"]].to_parquet("../data/preds_xgb.parquet", index=False)
print("saved ../data/preds_xgb.parquet:", f"{len(res):,}", "rows")

## Reading the results honestly

- XGBoost beats the lazy opponents almost everywhere, and the gain shows up in every block.
- On the five big stocks, compare it with **GARCH scaled** in Cell 10. If the two are close, that is an honest finding: a pooled model with many clues is not magic, and a good one-stock formula is competitive.
- None of these models can predict a surprise. A sudden news shock will always be missed by all of them. What they get right is the *normal* rhythm of calm and stormy spells.

## Your turn

1. In Cell 3 and Cell 6, change `max_depth=4` to `max_depth=2` (each advisor asks fewer questions) and re-run Cells 3 to 9. Does the score get better or worse?
2. In Cell 1's list, remove one clue (for example `vol_ratio`) by making a copy of `FEATURE_COLS` without it and using that in Cells 3 and 6. How much does the score change? What does that tell you about the clue?
3. Which clue mattered most in Cell 4? Is that what you would have guessed?
4. Explain back: **why do we choose the number of advisors using a practice slice of the old data, and not the real exam?** (Hint: think about what would happen to the exam score if we picked whatever setting scored best on the exam itself.)